# Lesson 1.6 — Task Decomposition Strategies & Session State / Resume / Fork

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Choose prompt chaining (fixed pipeline) vs dynamic adaptive decomposition
- Split large reviews into per-file passes plus a cross-file integration pass
- Resume named sessions and inform the agent about changed files
- Use fork_session to branch divergent approaches from a shared baseline

**You will produce:** A decomposition selector plus a session resume/fork plan with a changed-files notice.

## Concepts

### Task decomposition (1.6)
- **Prompt chaining (fixed sequential pipeline):** best for *predictable* multi-aspect reviews - e.g. analyze each file individually, then run a cross-file integration pass. Sample Question 12 confirms this for a 14-file PR with inconsistent single-pass results.
- **Dynamic adaptive decomposition:** best for *open-ended* investigation - map structure, find high-impact areas, then build a prioritized plan that adapts as dependencies are discovered (e.g. "add comprehensive tests to a legacy codebase").

### Session state, resumption, forking (1.7)
- **`--resume <session-name>`** continues a specific named prior conversation.
- **`fork_session`** branches independent explorations from a shared analysis baseline (e.g. compare two refactoring strategies).
- When resuming after code changes, **inform the agent which files changed** for targeted re-analysis instead of full re-exploration.
- When prior tool results are **stale**, starting a new session with a structured summary is *more reliable* than resuming with stale results.

> **Reference:** [Anthropic - Claude Code sessions & Agent SDK](https://docs.anthropic.com/en/docs/claude-code)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Choose a decomposition strategy

A predictable multi-file review uses prompt chaining; open-ended work uses dynamic planning.

In [ ]:
def choose_decomposition(task):
    predictable = task.get("well_scoped") and task.get("aspects_known")
    return "prompt_chaining" if predictable else "dynamic_adaptive"

pr_review = {"well_scoped": True, "aspects_known": True, "files": 14}
legacy_tests = {"well_scoped": False, "aspects_known": False}
print("14-file PR review:", choose_decomposition(pr_review))
print("legacy test coverage:", choose_decomposition(legacy_tests))

## Code-Along: Step 2 — Prompt chaining: per-file + integration pass

Avoid attention dilution by reviewing each file locally, then one cross-file pass.

In [ ]:
def chained_review(files):
    passes = [f"local-analysis:{f}" for f in files]
    passes.append("integration-pass:cross-file-data-flow")
    return passes

for p in chained_review(["stock.py", "ledger.py", "api.py"]):
    print(p)

## Code-Along: Step 3 — Resume and fork sessions (CLI model)

We model session commands. In Claude Code these map to `--resume <name>` and the Agent SDK `fork_session` option - confirm exact flags at https://docs.anthropic.com.

In [ ]:
def resume_cmd(session_name, changed_files=None):
    cmd = f"claude --resume {session_name}"
    if changed_files:
        note = "; ".join(changed_files)
        cmd += f'  # inform: these files changed -> re-analyze: {note}'
    return cmd

def fork_plan(baseline):
    return {"baseline": baseline,
            "branch_a": "fork_session -> strategy A (repository pattern)",
            "branch_b": "fork_session -> strategy B (active record)"}

print(resume_cmd("refund-audit", ["ledger.py"]))
print(json.dumps(fork_plan("shared-codebase-analysis"), indent=2))

## Exercise

Write `resume_or_restart(prior_results_stale, session_name, summary)` that returns a `--resume` command when results are valid, but returns a *fresh-start* plan injecting a structured `summary` when prior tool results are stale. Justify the choice in a comment, citing the exam's reliability guidance.

In [ ]:
# TODO: implement resume_or_restart(prior_results_stale, session_name, summary)
# TODO: return a fresh-start-with-summary plan when prior_results_stale is True
# TODO: return resume_cmd(...) otherwise

## Key Takeaways

- Prompt chaining for predictable multi-aspect reviews; dynamic decomposition for open-ended work.
- Split big reviews into per-file passes + a cross-file integration pass to avoid attention dilution.
- Use --resume for named sessions and tell the agent which files changed.
- fork_session branches divergent approaches from a shared baseline.
- When prior results are stale, start fresh with a structured summary instead of resuming.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.